# OpenPlaque — Left Coronary Bifurcation-Derived Parent Recovery v1
Fresh-baseline experiment. Runtime → Run all. The frozen master is never modified.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys
DRIVE_ROOT=Path('/content/drive/MyDrive/OpenPlaque')
OUTPUT_DIR=DRIVE_ROOT/'Left_Coronary_Bifurcation_Parent_Recovery_v1'
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
BRANCH='left-coronary-bifurcation-parent-recovery-from-main'
PIN='96e800a5269a80354b51af8428b6ffcd5ecc8bc1'
BASELINE='0593b453959f5a353d644267fbeef24b514ef4d7'
(OUTPUT_DIR/'notebook_started.json').write_text(json.dumps({'status':'STARTED','branch':BRANCH,'pin':PIN,'baseline':BASELINE},indent=2))
print('Output:',OUTPUT_DIR)


In [ ]:
REPO=Path('/content/OpenPlaque')
if REPO.exists(): shutil.rmtree(REPO)
subprocess.check_call(['git','clone','--branch',BRANCH,'--single-branch','https://github.com/pazzani/OpenPlaque.git',str(REPO)])
subprocess.check_call(['git','checkout',PIN],cwd=REPO)
HEAD=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
MERGE_BASE=subprocess.check_output(['git','merge-base','HEAD',BASELINE],cwd=REPO,text=True).strip()
assert HEAD==PIN,(HEAD,PIN)
assert MERGE_BASE==BASELINE,(MERGE_BASE,BASELINE)
print('Pinned science:',HEAD)
print('Exact frozen merge base:',MERGE_BASE)


In [ ]:
subprocess.check_call([sys.executable,'-m','pip','install','-q','.'],cwd=REPO)
for name in list(sys.modules):
    if name=='openplaque' or name.startswith('openplaque.'):
        del sys.modules[name]
import py_compile
py_compile.compile(str(REPO/'src/openplaque/left_coronary_bifurcation_parent_recovery_v1.py'),doraise=True)
sys.path.insert(0,str(REPO/'src'))
from openplaque.left_coronary_bifurcation_parent_recovery_v1 import synthetic_parent_recovery_self_test
print('Synthetic:',synthetic_parent_recovery_self_test())
subprocess.check_call([sys.executable,'-m','pytest','-q','tests/test_left_coronary_bifurcation_parent_recovery_v1.py'],cwd=REPO)


In [ ]:
required=[
 DRIVE_ROOT/'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.npy',
 DRIVE_ROOT/'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.json',
 DRIVE_ROOT/'Cache/Master_Coronary_Anatomy_Baseline_v2/master_anatomy_summary.json',
 DRIVE_ROOT/'Cache/LAD_Frozen_Proximal_Reacquisition_v1/combined_lad_centerline.csv',
 DRIVE_ROOT/'Left_Proximal_Trunk_Continuation_QC_v1/accepted_proximal_trunk_continuation_candidate.csv',
 DRIVE_ROOT/'Left_Proximal_Trunk_Continuation_QC_v1/summary.json',
 DRIVE_ROOT/'Joint_Three_Vessel_Template_Classifier_v1/candidate_04_source_path.csv',
 DRIVE_ROOT/'LCX_Distal_Reacquisition_v1_fixed/C7_extended_path.csv',
 DRIVE_ROOT/'LCX_Structural_Identity_Adjudication_v1/structural_identity_decision.json',
 DRIVE_ROOT/'PCAT_RCA_10_50/rca_centerline_smoothed_zyx.csv',
 DRIVE_ROOT/'TotalSegmentator_Cardiovascular_Cache_v1/heartchambers_highres/aorta.nii.gz',
]
missing=[str(p) for p in required if not p.exists()]
if missing: raise FileNotFoundError('Missing prerequisites:\n'+'\n'.join(missing))
(OUTPUT_DIR/'preflight_complete.json').write_text(json.dumps({'status':'PASS','n_required':len(required),'pin':PIN},indent=2))
print('Preflight PASS:',len(required),'inputs')


In [ ]:
os.chdir(REPO)
for name in list(sys.modules):
    if name=='openplaque' or name.startswith('openplaque.'):
        del sys.modules[name]
from openplaque.left_coronary_bifurcation_parent_recovery_v1 import run
result=run(str(DRIVE_ROOT),str(OUTPUT_DIR))
s=result['summary']
print('STATUS:',s.get('status'))
print('COMMON-TRUNK OVERLAP:',s.get('junction',{}).get('prox_common_overlap_pass'))
print('C6/C7 PARENT CONTROL:',s.get('junction',{}).get('control_pass'))
print('DAUGHTER ANGLE DEG:',s.get('junction',{}).get('daughter_angle_deg'))
print('TARGET ACCEPTED ARC:',s.get('target_best',{}).get('accepted_arc_mm'))
print('TARGET PLANE PASS:',s.get('target_best',{}).get('accepted_plane_pass_fraction'))
print('AORTA PROGRESS MM:',s.get('posthoc',{}).get('maximum_aorta_progress_mm'))
print('AORTA MIN DIST MM:',s.get('posthoc',{}).get('minimum_aorta_distance_mm'))
print('RCA ENDPOINT SEP MM:',s.get('posthoc',{}).get('endpoint_distance_to_known_RCA_mm'))
print('REPORT:',result['report'])
print('ZIP:',result['zip'])
